# 🎵 ANIMA — Microtonal Chord Generation
Generate 53-TET microtonal chord progressions using the trained GPT-2 model.

**Workflow:**
1. Load model + EigenSpace prior + tokenizer
2. Configure conditioning (TYPE, first chord)
3. Generate
4. Analyze chord quality
5. Export to MPE MIDI

In [ ]:
import sys, os, time, importlib
import numpy as np
import torch
from pathlib import Path
from IPython.display import Audio, display

SRC_DIR = Path('.').resolve()
ROOT_DIR = SRC_DIR.parent
sys.path.insert(0, str(SRC_DIR))

import play_mpe as pm
import midi_viz as mv

print(f'Root: {ROOT_DIR}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

## 1. Load Model

In [ ]:
import generate as gen
importlib.reload(gen)

# Load vocabulary
VOCAB_PATH = ROOT_DIR / 'dataset' / 'tokenized' / 'vocab.json'
vocab = gen.Vocabulary(str(VOCAB_PATH))
print(f'Vocabulary: {len(vocab)} tokens')

# Load model from best checkpoint
CHECKPOINT_PATH = ROOT_DIR / 'checkpoints' / 'best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, ckpt = gen.load_checkpoint(str(CHECKPOINT_PATH), device, vocab.vocab_size)

# Load EigenSpace computer
from eigenspace import EigenSpaceComputer
eigen_computer = EigenSpaceComputer(normalize_diss=True)

# Load EigenSpace prior (empirical distribution from training data)
TOKENIZED_DIR = ROOT_DIR / 'dataset' / 'tokenized'
eigen_prior = gen.EigenSpacePrior(str(TOKENIZED_DIR), max_seqs=2000)

# Load tokenizer (for proper MPE MIDI export with RPN setup)
from tokenizer import MPETokenizer
tokenizer = MPETokenizer()

print(f'\nDevice: {device}')
print(f'EigenSpace prior: {len(eigen_prior)} vectors')
print('Model loaded ✓')

## 2. Generation Settings
Adjust these parameters to control generation quality and conditioning.

**Conditioning controls:**
- `TYPE_LABEL`: Transformation type (e.g. `"0_major"`, `"2_subminor"`)
- `FIRST_CHORD`: Optional starting chord tokens to seed the progression
- `SEED`: Integer for reproducible results

In [ ]:
# ── Generation parameters ──
MAX_TOKENS    = 512
TEMPERATURE   = 0.91
TOP_K         = 10
TOP_P         = 0.99
SEED          = None

# EigenSpace recomputation mode — MUST be 'chord'
EIGEN_MODE    = 'chord'

# ── Conditioning ──
# Available types: 0_major, 0_minor, 1_minor, 1_neutral, 2_minor, 2_subminor,
#   3_major, 3_minor, 4_minor, 4_upmajor, 5_major_v2, 5_minor, 6_minor, 6_neutral_n
# Set to None for unconditional.
TYPE_LABEL    = '0_major'

# Available styles: jazz, bossa_samba, ballad, pop, rock, waltz, funk_soul, latin, blues, folk_country
# Set to None for unconditional.
STYLE_LABEL   = 'blues'

# Optional starting chord. Set to None for free generation.
# DvMvM7 — root D (pc=9), bass D3 (step 221), D4 (274), vM3rd (291), P5th (305), vM7th (322)
FIRST_CHORD   = 'CHORD_START DUR_4.0 ROOT_9 PV_221_6 PV_274_5 PV_291_4 PV_305_5 PV_322_4 CHORD_END'

# ── Audio rendering ──
PLAYBACK_SPEED = 1.2
WAVEFORM       = 'square' # 'sine', 'square', 'sawtooth', 'triangle'
REVERB         = 33
MIDI_TEMPO     = 160
SAVE_AUDIO     = True   # Save .wav files to disk for sharing10

# ── Output ──
OUTPUT_DIR  = ROOT_DIR / 'dataset' / 'generated'
MIDI_DIR    = OUTPUT_DIR / 'midi'
AUDIO_DIR   = OUTPUT_DIR / 'audio'
MIDI_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print('Settings configured ✓')
print(f'  Max tokens:   {MAX_TOKENS}')
print(f'  Temperature:  {TEMPERATURE}')
print(f'  Eigen mode:   {EIGEN_MODE}')
print(f'  Type:         {TYPE_LABEL or "unconditional"}')
print(f'  Style:        {STYLE_LABEL or "unconditional"}')
print(f'  First chord:  {"custom" if FIRST_CHORD else "free"}')
print(f'  Save audio:   {SAVE_AUDIO}')
print(f'  MIDI dir:     {MIDI_DIR}')
print(f'  Audio dir:    {AUDIO_DIR}')


## 3. Generate

In [ ]:
if SEED is not None:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    print(f'Seed: {SEED}')

# Build prompt: <start> [TYPE_<label>] [STYLE_<label>] [first chord tokens...]
prompt_tokens = ['<start>']

if TYPE_LABEL is not None:
    type_tok = f'TYPE_{TYPE_LABEL}'
    if type_tok in vocab.token_to_id:
        prompt_tokens.append(type_tok)
        print(f'Type conditioning:  {type_tok}')
    else:
        print(f'Unknown type "{TYPE_LABEL}", skipping')

if STYLE_LABEL is not None:
    style_tok = f'STYLE_{STYLE_LABEL}'
    if style_tok in vocab.token_to_id:
        prompt_tokens.append(style_tok)
        print(f'Style conditioning: {style_tok}')
    else:
        print(f'Unknown style "{STYLE_LABEL}" — valid: swing, bossa_samba, ballad, pop, rock, waltz, funk_soul, latin, blues, folk_country')

if FIRST_CHORD is not None:
    chord_toks = FIRST_CHORD.strip().split()
    unknown = [t for t in chord_toks if t not in vocab.token_to_id]
    if unknown:
        print(f'Unknown tokens in first chord: {unknown}')
        chord_toks = [t for t in chord_toks if t in vocab.token_to_id]
    prompt_tokens.extend(chord_toks)
    print(f'First chord: {" ".join(chord_toks)}')

prompt_ids = vocab.encode(prompt_tokens)
print(f'\nPrompt: {" ".join(prompt_tokens)}')
print(f'Generating {MAX_TOKENS} tokens...')

t0 = time.time()
gen_ids, gen_tokens = gen.generate_with_eigenspace(
    model, vocab, eigen_computer,
    prompt_ids=prompt_ids,
    max_new_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    device=device,
    recompute_interval=EIGEN_MODE,
    eigen_prior=eigen_prior,
)
dt = time.time() - t0
new_count = len(gen_ids) - len(prompt_ids)

chords = gen.extract_chords_summary(gen_tokens)
n_bars = sum(1 for t in gen_tokens if t == 'BAR')

print(f'\nGenerated {new_count} tokens in {dt:.2f}s ({new_count/dt:.0f} tok/s)')
print(f'  Chords: {len(chords)}')
print(f'  Bars:   {n_bars}')


## 4. Chord Quality Analysis
Verify the model produces real harmonic chords — not random notes.

In [ ]:
from collections import Counter

# ── Notes per chord distribution ──
pitch_counts = Counter(len(c['pitches']) for c in chords)

print('--- Notes per Chord ---')
print(f'Total chords: {len(chords)}\n')
for n_pitches in sorted(pitch_counts.keys()):
    pct = 100 * pitch_counts[n_pitches] / len(chords)
    bar = '#' * int(pct / 2)
    print(f'  {n_pitches} notes: {pitch_counts[n_pitches]:3d} ({pct:5.1f}%) {bar}')
avg = sum(k * v for k, v in pitch_counts.items()) / max(1, len(chords))
print(f'\n  Average: {avg:.2f} notes/chord (training avg: ~5.15)')

# ── ROOT consistency ──
root_matches = 0
root_total = 0
for c in chords:
    if c['root'] is not None and c['pitches']:
        root_total += 1
        bass_pc = min(c['pitches']) % 53
        if bass_pc == c['root']:
            root_matches += 1
print(f'\n--- ROOT Consistency ---')
print(f'  ROOT = bass pitch class: {root_matches}/{root_total} ({100*root_matches/max(1,root_total):.1f}%)')

# ── Duration distribution ──
dur_counts = Counter(c['duration'] for c in chords if c['duration'] is not None)
print(f'\n--- Duration Distribution ---')
for dur in sorted(dur_counts.keys()):
    pct = 100 * dur_counts[dur] / len(chords)
    bar = '#' * int(pct / 2)
    print(f'  DUR_{dur:4.1f}: {dur_counts[dur]:3d} ({pct:5.1f}%) {bar}')

# ── Interval analysis ──
CANONICAL_THIRDS = [11,12,13,14,15,16,17,18,19,20,22]
CANONICAL_FIFTHS = [26,31,35]

def classify_intervals(intervals_mod53):
    thirds = [i for i in intervals_mod53 if 10 <= i <= 22]
    fifths = [i for i in intervals_mod53 if 24 <= i <= 36]
    sevenths = [i for i in intervals_mod53 if 42 <= i <= 52]
    q3 = 'none'
    if thirds:
        t = min(thirds, key=lambda x: min(abs(x-c) for c in CANONICAL_THIRDS))
        if t <= 14: q3 = 'minor'
        elif t <= 17: q3 = 'neutral/major'
        else: q3 = 'augmented'
    q5 = 'none'
    if fifths:
        f = min(fifths, key=lambda x: min(abs(x-c) for c in CANONICAL_FIFTHS))
        if f <= 28: q5 = 'dim5'
        elif f <= 33: q5 = 'perf5'
        else: q5 = 'aug5'
    q7 = 'none'
    if sevenths:
        s = min(sevenths)
        if s <= 44: q7 = 'dim7'
        elif s <= 47: q7 = 'min7'
        else: q7 = 'maj7'
    return f'{q3}/{q5}/{q7}'

print(f'\n--- Chord Interval Analysis (first 12 chords) ---')
quality_counts = Counter()
for ci, c in enumerate(chords):
    pcs = [p % 53 for p in c['pitches']]
    root_pc = c['root'] if c['root'] is not None else (min(pcs) if pcs else 0)
    intervals = sorted(set([(pc - root_pc) % 53 for pc in pcs]))
    quality = classify_intervals(intervals)
    quality_counts[quality] += 1
    if ci < 12:
        print(f'  Chord {ci+1:2d}: root={root_pc:2d}  intervals={intervals}  dur={c["duration"]}  -> {quality}')

print(f'\n--- Quality Summary (all {len(chords)} chords) ---')
for q, cnt in quality_counts.most_common():
    print(f'  {q:30s}: {cnt:3d} ({100*cnt/len(chords):5.1f}%)')

# ── EigenSpace verification ──
eigen_vals = eigen_computer.compute_for_tokens(gen_tokens)
cs_positions = [i for i, t in enumerate(gen_tokens) if t == 'CHORD_START']
cs_eigens = eigen_vals[cs_positions]
unique_eigens = len(set(tuple(e) for e in cs_eigens))
print(f'\n--- EigenSpace ---')
print(f'  Unique eigenspace vectors: {unique_eigens}/{len(cs_positions)} chords')
print(f'  a (3rd):  [{cs_eigens[:,0].min():.3f}, {cs_eigens[:,0].max():.3f}]')
print(f'  b (5th):  [{cs_eigens[:,1].min():.3f}, {cs_eigens[:,1].max():.3f}]')
print(f'  g (7th):  [{cs_eigens[:,2].min():.3f}, {cs_eigens[:,2].max():.3f}]')
print(f'  d (root): [{cs_eigens[:,3].min():.3f}, {cs_eigens[:,3].max():.3f}]')

# ── Readable format ──
print(f'\n--- Readable Sequence (first 150 tokens) ---')
print(gen.format_as_readable(gen_tokens[:150]))

## 5. Export to MPE MIDI
Uses the tokenizer's proper MPE export with RPN pitch bend range setup.

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
midi_filename = f'generated_{timestamp}.mid'
midi_path = MIDI_DIR / midi_filename

# Decode tokens back to chord events, then write MPE MIDI with proper RPN
decoded_chords = tokenizer.decode(gen_tokens)

if decoded_chords:
    tokenizer.chords_to_midi(decoded_chords, str(midi_path), tpb=960, tempo_bpm=MIDI_TEMPO)
    n_notes = sum(len(c['notes']) for c in decoded_chords)
    print(f'Exported MPE MIDI: {midi_path.name}')
    print(f'  {len(decoded_chords)} chords, {n_notes} notes, {n_bars} bars')
    print(f'  Tempo: {MIDI_TEMPO} BPM, RPN pitch bend range: +/-2 semitones')
else:
    print('No chords decoded from the generated sequence.')

## 6. Visualize

In [ ]:
if midi_path.exists():
    fig = mv.visualize_midi(str(midi_path), speed=PLAYBACK_SPEED, max_duration=120)
    if fig:
        fig.update_layout(title=f'Generated: {midi_filename}', height=400)
        fig.show()

## 7. Play Audio
Render MIDI to audio for listening.

In [ ]:
if midi_path.exists():
    importlib.reload(pm)
    wav_path = AUDIO_DIR / midi_path.with_suffix('.wav').name if SAVE_AUDIO else None
    audio_data, sr = pm.render_mpe_to_audio_data(
        str(midi_path), speed=PLAYBACK_SPEED,
        waveform=WAVEFORM, reverb=REVERB,
        save_path=wav_path,
    )
    if audio_data is not None:
        display(Audio(audio_data, rate=sr))

## 8. Batch Generate
Generate multiple samples with the same conditioning.

In [ ]:
NUM_SAMPLES = 3

for i in range(NUM_SAMPLES):
    print(f'\n{"="*60}')
    print(f'Sample {i+1}/{NUM_SAMPLES}')
    print(f'{"="*60}')

    batch_prompt = ['<start>']
    if TYPE_LABEL is not None:
        type_tok = f'TYPE_{TYPE_LABEL}'
        if type_tok in vocab.token_to_id:
            batch_prompt.append(type_tok)
    if STYLE_LABEL is not None:
        style_tok = f'STYLE_{STYLE_LABEL}'
        if style_tok in vocab.token_to_id:
            batch_prompt.append(style_tok)
    if FIRST_CHORD is not None:
        chord_toks = [t for t in FIRST_CHORD.strip().split() if t in vocab.token_to_id]
        batch_prompt.extend(chord_toks)

    batch_prompt_ids = vocab.encode(batch_prompt)

    ids, tokens = gen.generate_with_eigenspace(
        model, vocab, eigen_computer,
        prompt_ids=batch_prompt_ids,
        max_new_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        top_k=TOP_K, top_p=TOP_P,
        device=device,
        recompute_interval=EIGEN_MODE,
        eigen_prior=eigen_prior,
    )

    sample_chords = gen.extract_chords_summary(tokens)
    sample_bars = sum(1 for t in tokens if t == 'BAR')
    print(f'  {len(sample_chords)} chords, {sample_bars} bars, {len(ids)} tokens')
    print(gen.format_as_readable(tokens[:100]))

    sample_path = MIDI_DIR / f'generated_{timestamp}_sample{i+1}.mid'
    decoded = tokenizer.decode(tokens)
    if decoded:
        tokenizer.chords_to_midi(decoded, str(sample_path), tpb=960, tempo_bpm=MIDI_TEMPO)
        print(f'  Exported: {sample_path.name}')

    if sample_path.exists():
        wav_path = AUDIO_DIR / sample_path.with_suffix('.wav').name if SAVE_AUDIO else None
        audio_data, sr = pm.render_mpe_to_audio_data(
            str(sample_path), speed=PLAYBACK_SPEED,
            waveform=WAVEFORM, reverb=REVERB,
            save_path=wav_path,
        )
        if audio_data is not None:
            display(Audio(audio_data, rate=sr))

print(f'\n{"="*60}')
print(f'All {NUM_SAMPLES} samples generated.')


## 9. Compare with Training Data

In [ ]:
import random

DATASET_MIDI_DIR = ROOT_DIR / 'dataset' / 'midi_files' / '53_tet_mpe'

if DATASET_MIDI_DIR.exists():
    if TYPE_LABEL is not None:
        type_dir = DATASET_MIDI_DIR / f'type_{TYPE_LABEL}'
    else:
        type_dirs = sorted(DATASET_MIDI_DIR.glob('type_*'))
        type_dir = random.choice(type_dirs) if type_dirs else DATASET_MIDI_DIR

    midi_files = sorted(type_dir.glob('*.mid'))
    if midi_files:
        ref_file = random.choice(midi_files)
        print(f'Reference file: {ref_file.relative_to(ROOT_DIR)}')

        fig = mv.visualize_midi(str(ref_file), speed=PLAYBACK_SPEED, max_duration=60)
        if fig:
            fig.update_layout(
                title=f'Training Data - {ref_file.parent.name}/{ref_file.stem}',
                height=400,
            )
            fig.show()

        audio_data, sr = pm.render_mpe_to_audio_data(
            str(ref_file), speed=PLAYBACK_SPEED,
            waveform=WAVEFORM, reverb=REVERB,
        )
        if audio_data is not None:
            display(Audio(audio_data, rate=sr))
    else:
        print(f'No MIDI files found in {type_dir}')
else:
    print(f'Dataset MIDI path not found: {DATASET_MIDI_DIR}')